# Parte 2: Modelagem Preditiva

Nesta segunda etapa do projeto, o foco transita da análise estatística e processamento de sinais para o campo do Aprendizado de Máquina. O objetivo é desenvolver e comparar diferentes modelos de classificação capazes de diagnosticar automaticamente o estado operacional do grupo motopropulsor baseando-se na telemetria.

Para abranger diferentes abordagens de reconhecimento de padrões, o trabalho avalia quatro algoritmos distintos, cada um sob a responsabilidade de um integrante da equipe:
- **Regressão Logística**
- **Support Vector Machine (SVM)**
- **Floresta Aleatória (Random Forest)**
- **K-Nearest Neighbors (KNN)**

Esta seção avalia o modelo sob a visão da competição em que a equipe participa, convertendo os dados do algoritmo em impacto direto na pontuação da **SAE Brasil AeroDesign (Classe Micro)**.

In [ ]:
# Importações necessárias para a Parte 2 (Machine Learning)
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler, PolynomialFeatures
import seaborn as sns
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt

# Regressão Logística
---

## 1. Extração de Características 

A extração de características ocorreu mediante uma técnica de janelamento de 50 amostras. Esta abordagem evita a correlação excessiva entre exemplos parecidos no conjunto de dados. Através disso, extraíram-se medidas estatísticas e métricas de domínio temporal de cada janela.

In [ ]:
def transitory_stacionary(df_i):
    # Substitua pelo seu código real que extrai o regime permanente, caso necessário.
    return None, df_i.copy(), None

def extrair_features_por_janela(df, tamanho_janela=50, passo=50):
    amostras = []
    # Ajustado o range para garantir que o processamento respeite o passo definido
    for i in range(0, len(df) - tamanho_janela + 1, passo):
        bloco = df.iloc[i : i + tamanho_janela]
        
        features = {}
        for col in bloco.columns:
            valores = bloco[col].dropna()
            if valores.empty:
                continue
            
            # Features estatísticas básicas
            features[f"{col}_Media"] = valores.mean()
            features[f"{col}_Mediana"] = valores.median()
            features[f"{col}_DesvioPadrao"] = valores.std()
            features[f"{col}_Skew"] = valores.skew()
            features[f"{col}_Kurtosis"] = valores.kurt()
            
            # Features avançadas
            q75, q25 = np.percentile(valores, [75 ,25])
            features[f"{col}_IQR"] = q75 - q25 
            
            rms = np.sqrt(np.mean(valores**2))
            features[f"{col}_RMS"] = rms
            features[f"{col}_Energia"] = np.sum(valores**2)
            
            # Features vitais para detecção de anomalias mecânicas
            val_max = valores.max()
            val_min = valores.min()
            features[f"{col}_PicoAPico"] = val_max - val_min 
            features[f"{col}_FatorCrista"] = np.max(np.abs(valores)) / rms if rms > 0 else 0
            
        amostras.append(pd.Series(features))
        
    return amostras

## 2. Preparação e Estruturação do Dataset

Nesta etapa, o algoritmo extrai as pastas com os ensaios originais coletados. O regime permanente é extraído, e em seguida, o janelamento é aplicado para criar a base que alimentará os algoritmos de *Machine Learning*.

In [ ]:
# PREPARAÇÃO DOS DADOS
base_dir = "C:/Users/Hugo Samuel/OneDrive/Desktop/MachineLearning_Trabalho1-main/dados_coletados"
pastas = ["Planilhas", "helice_quebrada_sintetica", "helice_desbalanceada_sintetica", "rotacao_invertida_sintetica", "normal_sintetica"]

lista_arquivos_com_classe = []

for p in pastas:
    caminho_p = os.path.join(base_dir, p)
    if os.path.exists(caminho_p):
        for arquivo in os.listdir(caminho_p):
            if arquivo.endswith(".xlsx"):
                nome_f = arquivo.lower()
                classe = 4
                if "normal" in nome_f: classe = 0
                elif "quebrada" in nome_f: classe = 1
                elif "desbalanceada" in nome_f: classe = 2
                elif "invertida" in nome_f: classe = 3
                
                if classe != 4:
                    lista_arquivos_com_classe.append((os.path.join(caminho_p, arquivo), classe))

# Divisão por arquivos inteiros para evitar data leakage
train_files, test_files = train_test_split(
    lista_arquivos_com_classe, 
    test_size=0.2, 
    random_state=42, 
    stratify=[c for _, c in lista_arquivos_com_classe]
)

def processar_arquivos(lista_arquivos):
    dados = []
    for caminho, classe in lista_arquivos:
        try:
            df_i = pd.read_excel(caminho)
            colunas_proibidas = [c for c in df_i.columns if any(s in c.lower() for s in ['potencia', 'rpm', 'tempo', 'celula3'])]
            df_i = df_i.drop(columns=colunas_proibidas, errors="ignore")
            
            _, df_1, _ = transitory_stacionary(df_i)
            amostras_janeladas = extrair_features_por_janela(df_1, tamanho_janela=50, passo=50)
            
            for linha in amostras_janeladas:
                linha["Classe"] = classe
                dados.append(linha)
        except Exception:
            pass
            
    return pd.DataFrame(dados).fillna(0)

df_train = processar_arquivos(train_files)
df_test = processar_arquivos(test_files)

X_train, y_train = df_train.drop(columns=["Classe"]), df_train["Classe"]
X_test, y_test = df_test.drop(columns=["Classe"]), df_test["Classe"]
print(f"Dataset final construído! Amostras de Treino: {len(X_train)} | Teste: {len(X_test)}")

## 3. Treinamento e Otimização de Limiar (Threshold Tuning)

Os dados foram divididos em 80% para treino e 20% para teste. O modelo baseou-se em um *Pipeline* contendo o `RobustScaler` (para atenuação de outliers) e a função `PolynomialFeatures` (para o mapeamento de fronteiras não-lineares).

Em vez de adotar o limiar de decisão padrão de 50%, implementou-se uma otimização por custo de erro (*Threshold Tuning*). O algoritmo avalia os impactos dos Falsos Positivos e Falsos Negativos e determina matematicamente a probabilidade ótima para acionar a manutenção preditiva.

In [ ]:
# =====================================================================
# TREINO DO MODELO E THRESHOLD TUNING
# =====================================================================
modelo = Pipeline([
    ("scaler", RobustScaler()), 
    ("poly", PolynomialFeatures(degree=2, interaction_only=True, include_bias=False)), 
    ("logreg", LogisticRegression(max_iter=5000, class_weight='balanced', C=10, solver='lbfgs', random_state=42))
])

modelo.fit(X_train, y_train)

# Extração das probabilidades brutas no conjunto de teste
probs = modelo.predict_proba(X_test)
prob_falha = np.sum(probs[:, 1:], axis=1)

limiares = np.linspace(0.1, 0.9, 50)
custos = []
melhor_tau = 0.5
menor_custo = float('inf')
pontos_maximos = 277
pts_desbalanceada = 138

# Testar todos os limiares para minimizar a perda de pontos na competição SAE
for tau in limiares:
    preds_tau = np.where(prob_falha >= tau, np.argmax(probs[:, 1:], axis=1) + 1, 0)
    
    pontos_cenario = 0
    for real, pred in zip(y_test, preds_tau):
        if pred == 0:
            if real == 0: pontos_cenario += pontos_maximos
            elif real == 1: pontos_cenario += 0
            elif real == 2: pontos_cenario += pts_desbalanceada
            elif real == 3: pontos_cenario += 0
        else: 
            if real == 0: pontos_cenario += 0 
            else: pontos_cenario += pontos_maximos
            
    # Custo Médio (Pontos perdidos por bateria)
    custo_total_perdido = (len(y_test) * pontos_maximos) - pontos_cenario
    custo_medio = custo_total_perdido / len(y_test) if len(y_test) > 0 else 0
    
    custos.append(custo_medio)
    
    if custo_medio < menor_custo:
        menor_custo = custo_medio
        melhor_tau = tau

print(f"-> Limiar (Threshold) Otimizado por Custo: {melhor_tau:.2f}\n")

# Aplicação final utilizando o limiar ótimo
y_pred = np.where(prob_falha >= melhor_tau, np.argmax(probs[:, 1:], axis=1) + 1, 0)

nomes_classes = ["Normal", "Quebrada", "Desbalanceada", "Invertida"]
classes_presentes = np.unique(y_test)
nomes_presentes = [nomes_classes[int(c)] for c in classes_presentes]

relatorio_dicionario = classification_report(y_test, y_pred, target_names=nomes_presentes, output_dict=True)
tabela_metricas = pd.DataFrame(relatorio_dicionario).transpose()

print("="*60)
print("   TABELA DE MÉTRICAS DE CLASSIFICAÇÃO (POR FALHA)")
print("="*60)
print(tabela_metricas[['precision', 'recall', 'f1-score', 'support']].round(3))
print("="*60)
print(f"ACURÁCIA GLOBAL DO MODELO: {accuracy_score(y_test, y_pred)*100:.2f}%")
print("="*60 + "\n")

## 4. Análise de Risco e Impacto na Pontuação SAE AeroDesign

Esta seção avalia os benefícios diretos do uso da classificação de Machine Learning (agora ajustada em prol da equipe) comparado com a decisão de voar às cegas.

In [ ]:
# =====================================================================
# CÁLCULO DE IMPACTO SAE AERODESIGN
# =====================================================================
pontos_sem_pesquisa, pontos_com_pesquisa = 0, 0

for real, pred in zip(y_test, y_pred):
    # Cenário 1: SEM pesquisa
    if real == 0: pontos_sem_pesquisa += pontos_maximos
    elif real == 1: pontos_sem_pesquisa += 0
    elif real == 2: pontos_sem_pesquisa += pts_desbalanceada
    elif real == 3: pontos_sem_pesquisa += 0
    
    # Cenário 2: COM pesquisa (Limiar Otimizado)
    if pred == 0:
        if real == 0: pontos_com_pesquisa += pontos_maximos
        elif real == 1: pontos_com_pesquisa += 0
        elif real == 2: pontos_com_pesquisa += pts_desbalanceada
        elif real == 3: pontos_com_pesquisa += 0
    else: 
        if real == 0: pontos_com_pesquisa += 0 
        else: pontos_com_pesquisa += pontos_maximos 

media_sem_pesquisa = pontos_sem_pesquisa / len(y_test) if len(y_test) > 0 else 0
media_com_pesquisa = pontos_com_pesquisa / len(y_test) if len(y_test) > 0 else 0
media_ideal = pontos_maximos

print("="*50)
print(" IMPACTO NA PONTUAÇÃO - SAE BRASIL AERODESIGN")
print("="*50)
print(f"Cenário 1 : Média de {media_sem_pesquisa:.1f} pontos por tentativa.")
print(f"Cenário 2 : Média de {media_com_pesquisa:.1f} pontos por tentativa.")
print(f"\n-> Com essa pesquisa, salva-se em média {media_com_pesquisa - media_sem_pesquisa:.1f} pontos por rodada!")
print("="*50 + "\n")

## 5. Visualização dos Resultados

In [ ]:
# =====================================================================
# PLOTAGEM DE RESULTADOS GRÁFICOS (Separados)
# =====================================================================

# Gráfico 1: Escolha do limiar por custo
plt.figure(figsize=(8, 5))
plt.plot(limiares, custos, color="#0408f1", linewidth=2.5, label='Custo de Erro')
plt.axvline(melhor_tau, color='black', linestyle='--', linewidth=2, label=f'Limiar Ótimo (tau={melhor_tau:.2f})')
plt.title("Otimização do Limiar por Custo (Perdas de Pontos na Competição)")
plt.xlabel("Limiar de Decisão (Probabilidade de Falha)")
plt.ylabel("Custo Médio (Pontos Perdidos por Bateria)") 
plt.legend()
plt.grid(True, linestyle=':', alpha=0.7)
plt.tight_layout()
plt.show()

# Gráfico 2: Matriz de Confusão
plt.figure(figsize=(8, 6))
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=nomes_presentes, yticklabels=nomes_presentes)
plt.title("Matriz de Confusão - Regressão Logística")
plt.xlabel("Previsto")
plt.ylabel("Condição Real")
plt.tight_layout()
plt.show()

# Gráfico 3: Impacto de Barras
plt.figure(figsize=(8, 6))
labels = ['Voo às Cegas\n', 'Manutenção Preditiva', 'Voo Ideal\n(Máximo Teórico)']
valores = [media_sem_pesquisa, media_com_pesquisa, media_ideal]
cores = ["#00f7ff", "#0e76ec", "#0b00a7"] 

barras = plt.bar(labels, valores, color=cores)
plt.ylim(0, pontos_maximos * 1.15)
plt.ylabel("Pontos Esperados por Bateria")
plt.title("Impacto da Pesquisa na Pontuação (Classe Micro)")

for b in barras:
    plt.annotate(f"{b.get_height():.1f} pts", 
                 xy=(b.get_x() + b.get_width() / 2, b.get_height()),
                 xytext=(0, 3), textcoords="offset points", 
                 ha='center', va='bottom', fontweight='bold', fontsize=14)

plt.tight_layout()
plt.show()
